# 67 — A/B Prompt Testing
**Goal:** Quantitatively compare prompt variants on a golden test set.

## 1. A/B Testing Framework

In [ ]:
class PromptABTest:
    def __init__(self, golden_set):
        self.golden = golden_set  # [(input, expected_output), ...]
    
    def evaluate(self, prompt_template, llm_fn):
        """Test a prompt variant against golden set."""
        results = []
        for input_text, expected in self.golden:
            output = llm_fn(prompt_template.format(input=input_text))
            exact = output.strip() == expected.strip()
            results.append({"input": input_text, "expected": expected, "got": output, "exact": exact})
        accuracy = sum(r["exact"] for r in results) / len(results)
        return {"accuracy": accuracy, "results": results}

# Simulated golden set
golden = [
    ("Responsible for ML models", "Developed ML models achieving 95% accuracy"),
    ("Was in charge of data pipeline", "Built automated data pipeline reducing processing time by 60%"),
    ("Helped with team projects", "Led cross-functional team delivering 3 major features"),
]

# Simulate A/B test
def variant_a(text):
    return text.replace("Responsible for", "Developed").replace("Was in charge of", "Built").replace("Helped with", "Led")

def variant_b(text):
    return text.replace("Responsible for", "Created").replace("Was in charge of", "Designed").replace("Helped with", "Managed")

test = PromptABTest(golden)
print("A/B Test Results:")
for name, fn in [("Variant A (aggressive verbs)", variant_a), ("Variant B (conservative)", variant_b)]:
    result = test.evaluate("Rewrite: {input}", fn)
    print(f"  {name}: {result['accuracy']:.0%} accuracy on golden set")
    for r in result['results']:
        if not r['exact']:
            print(f"    ✗ '{r['input'][:30]}...' -> '{r['got'][:30]}...' (expected '{r['expected'][:30]}...')")

## 2. Statistical Significance

In [ ]:
from scipy import stats
import numpy as np

# Example: 10 runs each of two prompt variants
variant_a_scores = [0.85, 0.87, 0.86, 0.84, 0.88, 0.85, 0.86, 0.87, 0.85, 0.86]
variant_b_scores = [0.82, 0.83, 0.81, 0.84, 0.82, 0.83, 0.81, 0.82, 0.83, 0.82]

t_stat, p_value = stats.ttest_ind(variant_a_scores, variant_b_scores)
print(f"A/B Statistical Test:")
print(f"  Variant A mean: {np.mean(variant_a_scores):.3f}")
print(f"  Variant B mean: {np.mean(variant_b_scores):.3f}")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")
print(f"  {'✓ Statistically significant (p<0.05)' if p_value < 0.05 else 'Not significant — difference may be noise'}")

## Summary: A/B test prompts against golden datasets. Use statistical tests to confirm significance.